# ML-07 · Baseline Action Score and Top-10 Review

**Lane: Refresh / Content Opportunity Scoring.** The queue helps an editor decide which pages to inspect first. This baseline uses the anonymized starter snapshot. The outcome is a drop of more than 20% in search impressions between two adjacent 30-day windows, a proxy for review priority. Measuring the effect of an actual refresh would require intervention records.

**Run:** From a checkout, run all cells using the checked-in starter CSV. In a fresh Google Colab runtime, the first code cell fetches the same starter CSV from this repository and checks its checksum. The generated queue remains in the runtime's `work/outputs/` directory.

## 1. Signal checks and rule reasoning

Score at the boundary between the previous and latest 30-day windows. Use previous-window impressions as the visibility signal, and subtract 30 days from snapshot content age to recover age at that boundary. Require a full 30-day observation window since creation and positive previous impressions. Evaluate the two hypotheses against the latest window.

- **Age and staleness:** Older content may be more likely to decline. Age measures time since creation and serves as a rough freshness proxy. Update recency at the scoring date cannot be recovered for all pages from the snapshot's last-update field.
- **Volume:** Higher past exposure might indicate a higher rate of later decline. Exposure is related to the volume signal behind FlyRank's quick-win flag. This exercise uses observed impressions as the exposure measure.

These are descriptive buckets from one snapshot. The label shares the previous-window denominator with the exposure signal, so the volume association may partly reflect how the decline proxy is defined.

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

relative_input = Path("data/raw/content_refresh_anonymized.csv")
here = Path.cwd()
root = next((p for p in (here, *here.parents) if (p / relative_input).is_file()), None)
if root is None:
    from importlib.util import find_spec
    from hashlib import sha256
    from urllib.request import urlopen

    if find_spec("google") is None or find_spec("google.colab") is None:
        raise FileNotFoundError(f"Run inside the starter repository containing {relative_input}")

    root = here
    data_url = (
        "https://raw.githubusercontent.com/mhv-codes/flyrank-ml-internship/"
        "48d0dac47169e9f14b106037938d414b16529aaf/"
        f"{relative_input}"
    )
    with urlopen(data_url, timeout=60) as response:
        starter_csv = response.read()
    expected_sha256 = "c43bdac4eccfa17fcd8a33974fe36f2c998c03a3ae3af8d80cf712abab5d6396"
    if sha256(starter_csv).hexdigest() != expected_sha256:
        raise ValueError("Downloaded starter CSV does not match the verified repository copy.")
    input_path = root / relative_input
    input_path.parent.mkdir(parents=True, exist_ok=True)
    input_path.write_bytes(starter_csv)

data = pd.read_csv(root / relative_input, usecols=[
    "content_id", "client_id", "content_age_days",
    "impressions_prev_30d", "impressions_last_30d",
])
keys = ["client_id", "content_id"]
assert data[keys].notna().all().all()
assert not data.duplicated(keys).any()
data["age_at_cutoff"] = data["content_age_days"] - 30
eligible = data.loc[data["age_at_cutoff"].ge(30) & data["impressions_prev_30d"].gt(0)].copy()
assert eligible["impressions_last_30d"].notna().all()
outcome = (5 * eligible["impressions_last_30d"]).lt(4 * eligible["impressions_prev_30d"])

def bucket_table(values, edges, names):
    buckets = pd.cut(values, bins=edges, labels=names)
    assert buckets.notna().all()
    return outcome.groupby(buckets, observed=True).agg(n="size", decline_rate="mean").round(4)

print(f"Snapshot rows: {len(data):,}. Eligible rows: {len(eligible):,}.")
print("Age at cutoff (days)")
display(bucket_table(eligible["age_at_cutoff"], [29, 89, 179, 364, float("inf")],
                     ["30–89", "90–179", "180–364", "365+"]))
print("Previous-window impressions")
display(bucket_table(eligible["impressions_prev_30d"], [0, 99, 499, 1999, float("inf")],
                     ["1–99", "100–499", "500–1,999", "2,000+"]))

,n,decline_rate
age_at_cutoff,,
30–89,4935,0.6738
90–179,6424,0.7298
180–364,9601,0.6021
365+,5652,0.4367


,n,decline_rate
impressions_prev_30d,,
1–99,8602,0.6013
100–499,6891,0.6281
"500–1,999",5855,0.6422
"2,000+",5264,0.5703


Snapshot rows: 30,000. Eligible rows: 26,612.
Age at cutoff (days)
Previous-window impressions


**Age verdict: OPPOSITE.** The two older buckets have lower observed decline rates (60.2% and 43.7%) than both younger buckets (67.4% and 73.0%). Leave age out of the score. The minimum age is only a coverage condition.

**Volume verdict: MIXED.** Decline increases from 60.1% to 64.2% across the first three buckets, then drops to 57.0% in the largest bucket. Volume can still measure how much previous exposure is at stake.

**Rule:** Queue an eligible page if it had at least 500 impressions in the previous window. Its score equals those impressions. Sort highest first and break ties by pseudonymous IDs for reproducibility. Each row gets the reason code `HIGH_PRIOR_EXPOSURE` and action `REVIEW_FOR_REFRESH`. The 500-impression floor is an editorial screening choice and has not been optimized for predictive accuracy.

## 2. Write the ranked queue and measure the baseline

In [2]:
score_column = "impressions_prev_30d"
queue = (eligible.loc[eligible[score_column].ge(500), keys + [score_column]]
         .sort_values([score_column, *keys], ascending=[False, True, True])
         .rename(columns={score_column: "score"})
         .reset_index())
queue.insert(0, "rank", queue.index + 1)
queue["reason_code"] = "HIGH_PRIOR_EXPOSURE"
queue["action"] = "REVIEW_FOR_REFRESH"

output_dir = root / "work/outputs"
output_dir.mkdir(parents=True, exist_ok=True)
queue.drop(columns="index").to_csv(output_dir / "baseline_action_score.csv", index=False)

metrics = {
    "input_rows": len(data),
    "eligible_rows": len(eligible),
    "queued_rows": len(queue),
    "minimum_prior_impressions": 500,
    "eligible_decline_base_rate": float(outcome.mean()),
}
for k in (10, 50):
    assert len(queue) >= k
    metrics[f"precision_at_{k}"] = float(outcome.loc[queue.head(k)["index"]].mean())
(output_dir / "baseline_action_score_metrics.json").write_text(
    json.dumps(metrics, indent=2) + "\n", encoding="utf-8")
print(json.dumps(metrics, indent=2))

{
  "input_rows": 30000,
  "eligible_rows": 26612,
  "queued_rows": 11119,
  "minimum_prior_impressions": 500,
  "eligible_decline_base_rate": 0.6110777093040733,
  "precision_at_10": 0.7,
  "precision_at_50": 0.5
}


## 3. Top-10 editorial review

All actions mean opening an editorial review. The stated failure conditions are questions for an editor to investigate using page context. These hypotheses were assigned during review, without inspection of raw URLs, queries, or page histories.

In [3]:
caveats = [
    "A short-lived seasonal event may account for much of the exposure.",
    "A temporary branded campaign may explain the previous-window volume.",
    "Search demand for the topic may already have fallen across comparable pages.",
    "High impressions could come from positions too low for a refresh to move clicks.",
    "The page may already satisfy intent, with impressions spread across unrelated queries.",
    "The relatively young page may still be settling after launch.",
    "An indexing or canonical issue could require a technical fix before editorial work.",
    "Another page may already target the same intent, making consolidation more appropriate.",
    "Prior exposure could come mainly from an outdated event with little continuing demand.",
    "At 96 days old at the cutoff, the page may still be gathering stable search coverage.",
]
top = queue.head(10)
review = top[["rank", "action"]].copy()
review["why"] = [
    f"{score:,} prior impressions; {age} days old at cutoff"
    for score, age in zip(top["score"], eligible.loc[top["index"], "age_at_cutoff"])
]
review["what_would_make_it_wrong"] = caveats
for row in review.itertuples(index=False):
    print(f"{row.rank}. {row.action} | {row.why} | {row.what_would_make_it_wrong}")

1. REVIEW_FOR_REFRESH | 218,786 prior impressions; 507 days old at cutoff | A short-lived seasonal event may account for much of the exposure.
2. REVIEW_FOR_REFRESH | 177,601 prior impressions; 415 days old at cutoff | A temporary branded campaign may explain the previous-window volume.
3. REVIEW_FOR_REFRESH | 174,235 prior impressions; 415 days old at cutoff | Search demand for the topic may already have fallen across comparable pages.
4. REVIEW_FOR_REFRESH | 164,079 prior impressions; 332 days old at cutoff | High impressions could come from positions too low for a refresh to move clicks.
5. REVIEW_FOR_REFRESH | 161,284 prior impressions; 415 days old at cutoff | The page may already satisfy intent, with impressions spread across unrelated queries.
6. REVIEW_FOR_REFRESH | 160,641 prior impressions; 123 days old at cutoff | The relatively young page may still be settling after launch.
7. REVIEW_FOR_REFRESH | 147,889 prior impressions; 452 days old at cutoff | An indexing or canonical 

## 4. Weak picks and limits

Ranks 6 and 10 are weaker freshness candidates because their content is relatively young. Verify whether the pages need a content change before allocating editing time. Exposure alone cannot diagnose technical issues, changes in search demand, or whether an update would help.

Precision@10 was 70% and precision@50 was 50%, against a 61.1% eligible-population decline rate. The larger review budget shows why the baseline should remain a comparison point. These are exploratory measurements from the same snapshot used for the signal checks. For the next model, freeze this rule, evaluate it on the same held-out rows and outcome as the model, and use comparable queue sizes.

## 5. Self-check

In [4]:
written = pd.read_csv(output_dir / "baseline_action_score.csv")
assert len(written) == len(queue)
assert written["rank"].tolist() == list(range(1, len(written) + 1))
assert written["score"].is_monotonic_decreasing
assert written["reason_code"].unique().tolist() == ["HIGH_PRIOR_EXPOSURE"]
assert written["action"].unique().tolist() == ["REVIEW_FOR_REFRESH"]
assert len(review) == 10 and review["what_would_make_it_wrong"].str.len().gt(0).all()
assert set(written.columns) == {"rank", *keys, "score", "reason_code", "action"}
print("Queue schema, order, reason code, action, and ten editorial reviews verified.")

Queue schema, order, reason code, action, and ten editorial reviews verified.
